# MABUC vertical slice — every layer of causalrl

This notebook walks through the Multi-Armed Bandit with Unobserved Confounders (MABUC),
the founding example of causal RL (Bareinboim, Forney, Pearl 2015). It exercises every
layer of the library: the structural causal model (`scm`), the environment (`envs`), the
agents (`agents`), and evaluation (`eval`).

**The puzzle:** both arms have identical interventional means `E[Y|do(X=a)] = 0.5`, so an
agent that only reasons about interventions cannot tell them apart. But conditioning on the
observed *intuition* reveals the lucky arm — so a causal agent earns ~0.75.

## Layer 0–2: the structural causal model

Under `do(X=a)`, the arms are indistinguishable — both interventional means are ~0.5.

In [1]:
from causalrl.envs.suite.mabuc import build_mabuc_scm

scm = build_mabuc_scm()
print("E[Y|do(X=0)] =", scm.do({"X": 0.0}).see(20000, seed=0)["Y"].mean().item())
print("E[Y|do(X=1)] =", scm.do({"X": 1.0}).see(20000, seed=1)["Y"].mean().item())

E[Y|do(X=0)] = 0.5001999735832214
E[Y|do(X=1)] = 0.5009499788284302


## Layers 3–4: agents and evaluation

The causal agent conditions on intuition and converges to ~0.75; the naive agent ignores it
and stays near 0.50.

In [2]:
from causalrl.agents.bandits import CausalThompsonSampling, NaiveThompsonSampling
from causalrl.envs.suite.mabuc import MABUCEnv
from causalrl.eval.metrics import cumulative_regret


def run(agent, n=8000):
    env = MABUCEnv(seed=1)
    obs, _ = env.reset(seed=1)
    rewards = []
    for _ in range(n):
        a = agent.act(obs)
        _, r, _, _, _ = env.step(a)
        agent.update(obs, a, r)
        obs, _ = env.reset()
        rewards.append(r)
    return rewards


causal = run(CausalThompsonSampling(2, 2, seed=0))
naive = run(NaiveThompsonSampling(2, seed=0))
print("causal avg:", sum(causal) / len(causal))
print("naive avg:", sum(naive) / len(naive))
print("causal regret:", cumulative_regret(causal, optimal_per_step=0.75))
print("naive regret:", cumulative_regret(naive, optimal_per_step=0.75))

causal avg: 0.756
naive avg: 0.50925
causal regret: -48.0
naive regret: 1926.0
